# 01 · Diccionario de datos — Adopción de IA en empresas (2023–2026)

Este cuaderno recalcula el perfil automático del diccionario sobre las **150.000 filas** de `ai_company_adoption.csv`, le agrega a mano qué preguntas usa cada variable, los valores imposibles y la decisión tomada, y guarda el resultado en `diccionario_ai_v2.csv`.

**Fuente:** Kaggle, *Global AI Adoption and Workforce Impact Dataset* (mohankrishnathalla): https://www.kaggle.com/datasets/mohankrishnathalla/global-ai-adoption-and-workforce-impact-dataset?select=ai_company_adoption.csv. La licencia y cómo se generaron los datos hay que confirmarlas en esa página y citarlas en el `README.md`.

**Unidad de análisis:** una fila es una **empresa en un trimestre** (empresa-trimestre). Se analizan todas las filas.

**Preguntas de análisis (versión de trabajo):**

1. **P1** · Región con mayor tasa promedio de adopción de IA (`ai_adoption_rate`, `region`).
2. **P2** · Sector industrial con mayor adopción promedio (`ai_adoption_rate`, `industry`).
3. **P3** · Tamaño de empresa y adopción promedio (`ai_adoption_rate`, `company_size`).
4. **P4** · Horas de capacitación y cambio de productividad (`ai_training_hours`, `productivity_change_percent`).
5. **P5** · Automatización de tareas y horas ahorradas por semana (`task_automation_rate`, `time_saved_per_week`).
6. **P6** · Adopción de IA y puestos desplazados (`ai_adoption_rate`, `jobs_displaced`).
7. **P7** · Presupuesto en IA y crecimiento de ingresos (`ai_budget_percentage`, `revenue_growth_percent`).
8. **P8** · Madurez en IA y reducción de costes (`ai_maturity_score`, `cost_reduction_percent`).
9. **P9** · Gestión de riesgos y cumplimiento normativo (`ai_risk_management_score`, `regulatory_compliance_score`).
10. **P10** · Trabajo remoto y satisfacción de los empleados (`remote_work_percentage`, `employee_satisfaction_score`).

Todas las preguntas se refieren al período 2023–2026.

## 1. Lectura

El archivo se lee tal cual. Se desactivan los NA automáticos de pandas porque `none` es una **etapa de adopción** válida (`ai_adoption_stage`), no un faltante.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

pd.set_option('display.width', 200)
pd.set_option('display.max_columns', 20)

# Sube desde donde esté el cuaderno hasta encontrar la carpeta que tiene `datos/`
RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'datos').is_dir())
RUTA = RAIZ / 'datos' / 'crudos' / 'ai_company_adoption.csv'
RUTA_DIC = RAIZ / 'diccionario_datos_ai_company_adoption.xlsx'
assert RUTA.exists(), f'no se encuentra {RUTA.relative_to(RAIZ)}'
assert RUTA_DIC.exists(), f'no se encuentra {RUTA_DIC.relative_to(RAIZ)}'

df = pd.read_csv(RUTA, keep_default_na=False, na_values=[''])
print('filas:', len(df), '| columnas:', df.shape[1], '| celdas nulas:', int(df.isna().sum().sum()))
print('response_id único:', df['response_id'].is_unique, '| empresas (company_id):', df['company_id'].nunique())
assert len(df) == 150_000 and df.shape[1] == 43

filas: 150000 | columnas: 43 | celdas nulas: 0
response_id único: True | empresas (company_id): 10000


## 2. Estructura de panel

`company_id` identifica a la **empresa**, no a la fila: cada empresa aparece en varios trimestres. Se revisa cuántas filas tiene cada una, qué atributos no cambian dentro de una empresa y cómo se reparten los registros en el tiempo.

In [2]:
filas_por_empresa = df.groupby('company_id').size()
print('filas por empresa (cantidad de empresas):')
print(filas_por_empresa.value_counts().sort_index().to_string())
assert filas_por_empresa.between(10, 16).all()

print('\nsaltos en la numeración de response_id (máximo − filas):', df['response_id'].max() - len(df))
assert df['response_id'].max() == 10_000 * 16   # 10.000 empresas x 16 trimestres posibles

# Atributos que no cambian dentro de una empresa
fijas = ['country', 'region', 'industry', 'company_size', 'company_founding_year',
         'num_employees', 'annual_revenue_usd_millions']
assert (df.groupby('company_id')[fijas].nunique() == 1).all().all()

print('\nregistros por año y trimestre:')
pd.crosstab(df['survey_year'], df['quarter'])

filas por empresa (cantidad de empresas):
10       5
11      31
12     130
13     583
14    1871
15    3804
16    3576

saltos en la numeración de response_id (máximo − filas): 10000

registros por año y trimestre:


quarter,Q1,Q2,Q3,Q4
survey_year,,,,
2023,9342,9381,9346,9387
2024,9370,9357,9356,9397
2025,9411,9345,9387,9381
2026,9354,9406,9403,9377


**Registro de decisiones #1:** la unidad de análisis es **empresa-trimestre** y se usan las **150.000 filas**, sin promediar por empresa ni fijar un año. En títulos y conclusiones se habla de "empresa-trimestre", no de "empresas" a secas.

**Registro de decisiones #2:** el archivo tiene **150.000 filas**, pero `response_id` llega hasta 160.000: la numeración tiene 10.000 saltos. Kaggle muestra los mismos 150.000 registros y el mismo rango de ids (el archivo también pesa lo mismo, 40,42 MB), así que viene así del dataset original y no de la descarga; la página no explica el motivo. El número coincide con 10.000 empresas × 16 trimestres posibles (2023 Q1 a 2026 Q4), y lo que sí está comprobado es que cada empresa tiene entre 10 y 16 filas: el panel no está balanceado. Se conservan las 150.000 filas y no se completan los trimestres que faltan.

**Registro de decisiones #3:** el año 2026 trae los cuatro trimestres con unas 9.400 filas cada uno, aunque el Q4 de 2026 todavía no había terminado cuando se armó este análisis (octubre de 2026). No hay forma de verificar a qué corresponden esos registros. Se conservan y se aclara en la ficha del dataset.

**Registro de decisiones #4:** `country`, `region`, `industry`, `company_size`, `company_founding_year`, `num_employees` y `annual_revenue_usd_millions` son **constantes dentro de cada empresa** (se verifica con el `assert`). El resto de las variables cambia de un trimestre a otro.

## 3. Perfil recalculado

Se parte del diccionario en Excel (descripción, tipo, unidad y rango) y se agrega el perfil calculado sobre el CSV.

In [3]:
dic = pd.read_excel(RUTA_DIC, sheet_name='Diccionario de datos').set_index('Nombre de variable')
assert list(dic.index) == list(df.columns)

num = df.select_dtypes('number').columns
dic['no nulos']            = df.notna().sum()
dic['% nulos']             = (100 * df.isna().mean()).round(2)
dic['únicos']              = df.nunique()
dic['mínimo']              = df[num].min()
dic['máximo']              = df[num].max()
dic['ceros']               = (df[num] == 0).sum()
dic['filas en el mínimo']  = pd.Series({c: int((df[c] == df[c].min()).sum()) for c in num})
dic['filas en el máximo']  = pd.Series({c: int((df[c] == df[c].max()).sum()) for c in num})
for c in ['no nulos', 'únicos', 'ceros', 'filas en el mínimo', 'filas en el máximo']:
    dic[c] = dic[c].astype('Int64')

dic.loc[num.drop(['response_id', 'survey_year']), ['mínimo', 'máximo', 'ceros', 'filas en el mínimo', 'filas en el máximo']]

,mínimo,máximo,ceros,filas en el mínimo,filas en el máximo
num_employees,10.00,19970.000,0,631,14
annual_revenue_usd_millions,1.01,9996.730,0,31,16
company_founding_year,1990.00,2022.000,0,4416,4611
company_age,1.00,36.000,0,1154,1106
ai_adoption_rate,0.00,100.000,896,896,1
years_using_ai,0.00,10.000,5198,5198,3685
num_ai_tools_used,1.00,6.000,0,32063,62
ai_projects_active,0.00,20.000,8414,8414,19
ai_training_hours,0.00,80.000,3043,3043,9
ai_budget_percentage,0.00,25.000,782,782,8


Hay variables que acumulan muchas filas **exactamente** en su mínimo o su máximo: por ejemplo `cost_reduction_percent` tiene 15.887 filas en 0, `productivity_change_percent` 9.027 en 0, `revenue_growth_percent` 6.106 en −5, `innovation_score` 7.766 en 40 y `ai_failure_rate` 5.237 en 40. Es lo que se ve cuando los valores están recortados a un rango. No son errores ni faltantes: se conservan como dato y se aclara en la ficha, junto con la duda sobre cómo se generaron los datos.

## 4. Qué pregunta usa cada variable

`survey_year` entra en todas porque cada pregunta fija el período 2023–2026.

In [4]:
USO = {
    'P1':  ['ai_adoption_rate', 'region'],
    'P2':  ['ai_adoption_rate', 'industry'],
    'P3':  ['ai_adoption_rate', 'company_size'],
    'P4':  ['ai_training_hours', 'productivity_change_percent'],
    'P5':  ['task_automation_rate', 'time_saved_per_week'],
    'P6':  ['ai_adoption_rate', 'jobs_displaced'],
    'P7':  ['ai_budget_percentage', 'revenue_growth_percent'],
    'P8':  ['ai_maturity_score', 'cost_reduction_percent'],
    'P9':  ['ai_risk_management_score', 'regulatory_compliance_score'],
    'P10': ['remote_work_percentage', 'employee_satisfaction_score'],
}
USO = {p: cols + ['survey_year'] for p, cols in USO.items()}
assert not [c for cs in USO.values() for c in cs if c not in dic.index]

dic['pregunta afectada'] = ''
for p, cols in USO.items():
    for c in cols:
        dic.loc[c, 'pregunta afectada'] = (dic.loc[c, 'pregunta afectada'] + ' ' + p).strip()

dic.loc[dic['pregunta afectada'] != '', ['pregunta afectada']]

,pregunta afectada
Nombre de variable,
survey_year,P1 P2 P3 P4 P5 P6 P7 P8 P9 P10
region,P1
industry,P2
company_size,P3
ai_adoption_rate,P1 P2 P3 P6
ai_training_hours,P4
ai_budget_percentage,P7
ai_maturity_score,P8
regulatory_compliance_score,P9


## 5. Valores imposibles

Se separan de los atípicos: un valor imposible no puede ocurrir; un atípico es raro pero posible. Un porcentaje de crecimiento puede ser negativo, así que `revenue_growth_percent` solo tiene un límite inferior de −100.

In [5]:
pct = ['ai_adoption_rate', 'ai_budget_percentage', 'remote_work_percentage', 'task_automation_rate',
       'ai_failure_rate', 'productivity_change_percent', 'cost_reduction_percent']

chequeos = {}
for c in pct:
    chequeos[f'{c} fuera de 0–100'] = ~df[c].between(0, 100)
chequeos['revenue_growth_percent menor a −100'] = df['revenue_growth_percent'] < -100
chequeos['time_saved_per_week mayor a 168 h (una semana)'] = df['time_saved_per_week'] > 168
chequeos['company_age distinta de survey_year − company_founding_year'] = (
    df['company_age'] != df['survey_year'] - df['company_founding_year'])
chequeos['years_using_ai mayor a company_age'] = df['years_using_ai'] > df['company_age']
for c in ['jobs_displaced', 'jobs_created', 'reskilled_employees']:
    chequeos[f'{c} mayor a num_employees'] = df[c] > df['num_employees']

conteo = pd.Series({k: int(v.sum()) for k, v in chequeos.items()}, name='filas imposibles')
print(conteo.to_string())
assert conteo.sum() == 0, 'hay valores imposibles: decidir qué hacer antes de seguir'

ai_adoption_rate fuera de 0–100                                0
ai_budget_percentage fuera de 0–100                            0
remote_work_percentage fuera de 0–100                          0
task_automation_rate fuera de 0–100                            0
ai_failure_rate fuera de 0–100                                 0
productivity_change_percent fuera de 0–100                     0
cost_reduction_percent fuera de 0–100                          0
revenue_growth_percent menor a −100                            0
time_saved_per_week mayor a 168 h (una semana)                 0
company_age distinta de survey_year − company_founding_year    0
years_using_ai mayor a company_age                             0
jobs_displaced mayor a num_employees                           0
jobs_created mayor a num_employees                             0
reskilled_employees mayor a num_employees                      0


In [6]:
IMPOSIBLE = {c: 'fuera de 0–100' for c in pct}
IMPOSIBLE['revenue_growth_percent'] = 'menor a −100'
IMPOSIBLE['time_saved_per_week'] = 'más de 168 (una semana tiene 168 h)'
IMPOSIBLE['company_age'] = 'distinta de survey_year − company_founding_year'
IMPOSIBLE['years_using_ai'] = 'mayor a company_age'
for c in ['jobs_displaced', 'jobs_created', 'reskilled_employees']:
    IMPOSIBLE[c] = 'mayor a num_employees'

dic['valores imposibles'] = ''
for c, texto in IMPOSIBLE.items():
    dic.loc[c, 'valores imposibles'] = texto

**Registro de decisiones #5:** ningún chequeo encuentra valores imposibles, así que no se modifica ningún dato. Si el archivo cambiara, el `assert` frena el cuaderno.

## 6. Relaciones entre variables

`ai_adoption_stage` y `years_using_ai` parecen derivar de otras columnas. Se comprueba antes de decidir cómo tratarlas.

In [7]:
orden_etapas = ['none', 'pilot', 'partial', 'full']
tramos = df.groupby('ai_adoption_stage')['ai_adoption_rate'].agg(['min', 'max', 'count']).loc[orden_etapas]
print(tramos.to_string())

# La etapa corresponde a tramos de la tasa de adopción: cortes alrededor de 10, 35 y 70
assert tramos.loc['none', 'max'] <= 10.01
assert 9.99 <= tramos.loc['pilot', 'min'] and tramos.loc['pilot', 'max'] <= 35.01
assert 34.99 <= tramos.loc['partial', 'min'] and tramos.loc['partial', 'max'] <= 70.01
assert tramos.loc['full', 'min'] >= 69.99

none_con_tasa = int(((df['ai_adoption_stage'] == 'none') & (df['ai_adoption_rate'] > 0)).sum())
print('\netapa none con tasa > 0:', none_con_tasa, 'de', int((df['ai_adoption_stage'] == 'none').sum()))

# years_using_ai es 0 exactamente cuando la etapa es none
assert ((df['years_using_ai'] == 0) == (df['ai_adoption_stage'] == 'none')).all()

                    min     max  count
ai_adoption_stage                     
none                0.0    9.99   5198
pilot              10.0   34.99  64317
partial            35.0   69.99  78800
full               70.0  100.00   1685

etapa none con tasa > 0: 4302 de 5198


**Registro de decisiones #6:** `ai_adoption_stage` no es una variable independiente: es `ai_adoption_rate` cortada en tramos (aprox. none ≤ 10, pilot 10–35, partial 35–70, full ≥ 70). Y `none` **no significa 0 % de adopción**: la mayoría de esas filas tiene una tasa mayor que 0. Además, `years_using_ai` es 0 exactamente cuando la etapa es `none`. Se conservan las tres columnas y se tiene en cuenta que dicen en parte lo mismo.

## 7. Decisión tomada por variable

Una línea corta por variable. Las decisiones que necesitan evidencia están en el **registro de decisiones** de arriba. `anotar` no repite un texto que ya está, así que se puede correr varias veces.

In [8]:
dic['decisión tomada'] = ''

def anotar(cols, texto):
    for c in cols:
        d = dic.loc[c, 'decisión tomada']
        if texto not in d:
            dic.loc[c, 'decisión tomada'] = texto if d == '' else d + ' | ' + texto

def fm(x):
    return f'{int(x):,}'.replace(',', '.')

def n(c, v):
    return fm((df[c] == v).sum())

def pc(c, v):
    return f'{100 * (df[c] == v).mean():.1f}'.replace('.', ',')

# --- Identificadores, tiempo y atributos fijos de la empresa
anotar(['response_id'], 'Identifica la fila; la numeración llega a 160.000 pero hay 150.000 filas (10.000 saltos); panel no balanceado, se conserva')
anotar(['company_id'], 'Identifica la empresa (10.000), no la fila; la unidad de análisis es empresa-trimestre y se usan todas las filas')
anotar(['survey_year', 'quarter'], 'Período 2023–2026 completo; el Q4 de 2026 no había terminado al armar el análisis (oct-2026): se conserva y se aclara en la ficha')
anotar(fijas, 'Constante dentro de cada empresa (verificado); sin cambios')
anotar(['company_age'], 'Igual a survey_year − company_founding_year en todas las filas; sin cambios')
anotar(['company_age_group'], 'Consistente con company_age; sin cambios')

# --- Variables de las preguntas
anotar(['region', 'industry', 'company_size'], 'Categórica sin nulos; los grupos tienen tamaños distintos, se reporta el n por grupo')
anotar(['ai_adoption_rate'], f'Rango 0–100 sin imposibles; {n("ai_adoption_rate", 0)} filas en 0 se conservan como dato')
anotar(['ai_training_hours'], f'{n("ai_training_hours", 0)} filas en 0 (sin capacitación) se conservan como dato')
anotar(['productivity_change_percent'], f'Piso en 0: {n("productivity_change_percent", 0)} filas ({pc("productivity_change_percent", 0)} %) en 0; se conservan, no son faltantes')
anotar(['task_automation_rate'], f'{n("task_automation_rate", 0)} filas en 0 se conservan como dato')
anotar(['time_saved_per_week'], f'{n("time_saved_per_week", 0)} filas en 0 se conservan como dato; sin imposibles')
anotar(['jobs_displaced'], f'Muy asimétrica (mediana {fm(df["jobs_displaced"].median())}, máximo {fm(df["jobs_displaced"].max())}); {n("jobs_displaced", 0)} ceros y los atípicos se conservan; ninguna fila supera num_employees')
anotar(['ai_budget_percentage'], f'Rango 0–25; {n("ai_budget_percentage", 0)} filas en 0 se conservan como dato')
anotar(['revenue_growth_percent'], f'{fm((df["revenue_growth_percent"] < 0).sum())} filas negativas (decrecimiento posible, no es error); piso en −5 con {n("revenue_growth_percent", -5)} filas')
anotar(['ai_maturity_score'], 'Escala 0–0,904 sin documentar: se usa para ordenar y comparar niveles, no se interpreta como porcentaje')
anotar(['cost_reduction_percent'], f'Piso en 0: {n("cost_reduction_percent", 0)} filas ({pc("cost_reduction_percent", 0)} %) en 0; se conservan, no son faltantes')
anotar(['ai_risk_management_score'], f'Escala 40–98 sin documentar; piso en 40 con {n("ai_risk_management_score", 40)} filas')
anotar(['regulatory_compliance_score'], 'Escala 40–100 sin documentar; sin cambios')
anotar(['remote_work_percentage'], 'Rango 0–100 sin imposibles; sin cambios')
anotar(['employee_satisfaction_score'], 'Escala 2,1–9,2 sin documentar; sin cambios')

# --- Variables ligadas a la etapa de adopción
anotar(['ai_adoption_stage'], 'Es ai_adoption_rate cortada en tramos (aprox. 10, 35 y 70); none no es 0 % de adopción; no es una variable independiente')
anotar(['years_using_ai'], 'Es 0 exactamente cuando la etapa es none; nunca supera company_age')

sin_decision = dic[(dic['decisión tomada'] == '') & (dic['pregunta afectada'] != '')]
assert len(sin_decision) == 0, f'variables de las preguntas sin decisión: {list(sin_decision.index)}'

# --- Resto de las variables: no entran en las 10 preguntas
anotar(['ai_primary_tool', 'ai_use_case'], 'Cambia entre trimestres de una misma empresa (atributo del registro); sin cambios; ninguna pregunta la usa')
anotar(['survey_source', 'data_collection_method'], 'Reparto casi uniforme entre sus 4 categorías; la procedencia real no se puede verificar; ninguna pregunta la usa')
resto = dic.index[dic['decisión tomada'] == '']
anotar(resto, 'Sin cambios; ninguna de las 10 preguntas la usa')
assert (dic['decisión tomada'] != '').all()

## 8. Verificación y guardado

In [9]:
salida = RAIZ / 'diccionario_ai_v2.csv'
dic.reset_index().to_csv(salida, index=False, encoding='utf-8-sig')

leido = pd.read_csv(salida)
assert len(leido) == 43 and leido['decisión tomada'].notna().all()
print('guardado:', salida.relative_to(RAIZ), '|', dic.shape[0], 'variables |', dic.shape[1], 'columnas')

guardado: diccionario_ai_v2.csv | 43 variables | 16 columnas
